# ISS_preprocessing of Leica TIFF and LIF files, Zeiss CZI files and Nikon ND2 files

This notebook guides you through the preprocessing of TIFF files, as autosaved or exported by Leica microscopes, Leica LIF files, Zeiss CZI files, and Nikon Nd2 files, with associated Metadata.
 

To use this notebook the you must have:
1) **Individual TIFF files**, autosaved or exported, directly from the Leica software. Each tiff file will represent a single plane of a single tile of a single channel, so thousands of individual TIFFs will be created in a typical experiment.
2) **A single LIF file** (containing single or multiple regions) **or multiple LIF files** (one for each region), saved directly from the Leica software.
3) **A single CZI file** (containing single or multiple regions) **or multiple CZI files** (one for each region), saved directly from the Zeiss software.
4) **A single ND2 file** (containing single or multiple regions) **or multiple ND2 files** (one for each region), saved directly from the Nikon software.

---

For our script to work, **two things are very important**:  

---

### **1. File naming structure:**
   
   It is crucial how the **tile** and **channel** are represented in the TIFF filenames.

   #### Autosaved TIFF files:
   An example file name is `TileScan 1--Stage99--Z42--C03.tif`, where
   
   - `TileScan` represents the ROI (everything **before the first `--`** will represent the region),

   - `Stage` represents the tile,
  
   - `Z` represent Z-plane,
  
   -  `C` represents the channel (always written with two digits, e.g. `C01`, `C02`, `C03`), and
  
   -  the `--`  sign acts as a separator (default on our Leica microscopes).

  ⚠️ **Important**:  
  Filenames must contain both `--Stage<tile>--` and `--C<channel>` to be parsed correctly. 

  <details>
<summary>🔎 Regex rules (click to expand)</summary>

- Tile: `--Stage(\d+)--`  
- Channel: `--C(\d{2})`  

</details>

---
  
  #### Exported TIFF files:
  An example file name `2024-10-24--12-31-35-538 Region 1_s00_z42_ch3.tif`, where 
  
  `2024-10-24--12-31-35-538 Region 1` represents the ROI (everything **before the first `_`** will represent the region),
  
  - `s` represents the tile,

  - `z` represent Z-plane,

  -  `ch` represents the channel, and

  -   the `_`  sign acts as a separator (default on our Leica microscopes).

  ⚠️ **Important**:  
  Filenames must contain both `_s<tile>_` and `_ch<channel>` to be parsed correctly.

  <details>
<summary>🔎 Regex rules (click to expand)</summary>

- Tile: `_s(\d+)_`  
- Channel: `_ch(\d+)`  

</details>

---

  #### 📂 LIF files 
  - There are **no constraints on naming** for LIF files.

---

  #### 📂 CZI files 
  - There are **no constraints on naming** for CZI files.

---

  #### 📂 ND2 files 
  - There are **no constraints on naming** for Nd2 files.

---

### **2. Cycle folders:**

For **TIFF, LIF, CZI and ND2 files**, it is essential that files for **each cycle are stored in separate folders** (the name of the folder is not important).  
This ensures the pipeline can correctly distinguish cycles during preprocessing.
  
Examples:

`'/path/to/Cycle1/your_file.tif'`


`'/path/to/Cycle2/your_file.lif'` 


`'/path/to/Cycle3/your_file.czi'`


`'/path/to/Cycle3/your_file.nd2'`


 






Now we import the necessary libraries and tools

In [2]:
import ISS_preprocessing.preprocessing as pp


## This notebook offers deconvolution using `RedLionFish/Deconwolf`

`RedLionFish` is a GPU-accelerated deconvolution tool and to run it **you need a CUDA-compatible GPU with functional NVIDIA drivers, cuDNN, etc...**
`Deconwolf` is a CPU-based deconvolution tool. The Deconwolf executable is a command-line tool and **needs to be installed before using, see deconwolf_install.txt.** You can skip this installation if using only `RedLionFish`.

To do image deconvolution, you will first need to know some parameters about your microscope. Take some time to carefully read the guide below and to collect some information before you attempt image deconvolution. This will save you a lot of computing time and frustration: using the wrong parameters will result in imaging artifacts, so be careful!

These parameters will be used to create a synthetic point-spread function (PSF). If you don't know what a PSF is and why it is important, please read here: ​​https://en.wikipedia.org/wiki/Point_spread_function

You will need to know:

`na` = numerical aperture of the used len

`m` = lens magnification

`ni0` = refraction index of the immersion medium

`res_lateral` = x,y resolution of the images 

`res_axial`: z-resolution of the stack. This is either the spacing between zplanes or the actual z resolution of your lens, whichever is larger. Remember that the actual Z resolution of your lens will match your experiment’s resolution only if you acquired the stacks at Nyquist conditions. (https://imb.uq.edu.au/research/facilities/microscopy/training-manuals/microscopy-online-resources/image-capture/nyquist-conditions)


## Introduce the PSF_metadata

Once you've collected the above information, you have to input it in the format indicated below. You can substitute the numbers with the actual values for your microscope. **The PSF is ONLY needed when doing deconvolution.**


In [3]:
#THIS IS FOR 5 COLOURS LEICA 20X
PSF_metadata = {'na':1.1,
'm':20,
'ni0':1.333,
'res_lateral':0.419,
'res_axial':0.859,
 'channels':{
 '0':{
    'wavelength':.809},
  '1':{
    'wavelength':.681},
  '2':{
    'wavelength':.555},
  '3':{
    'wavelength':.475},
  '4':{
    'wavelength':.390},
  '5':{
    'wavelength':.436}
     
 }
}

## Main function for Leica preprocessing
This function processes Leica-exported TIFF and LIF files and under the hood runs many functions that are made invisible for convenience. It takes as input 3D imaging stacks, performs deconvolution (optional), projects the 3D stacks to 2D images, aligns and stitches the images, and finally outputs retiled images with desired dimensions. Once if you figured out the right parameters for your specific microscope, this will be the smoother way of running preprocessing.

### Features

**The function is able to process MULTIPLE cycles at a time.** This means that the list of input directories contains paths to the cycles you want to process, and you must manually specify the corresponding list of cycles. 

**This function is able to handle multiple regions** in the input files, and project them accordingly.

### Parameters

`input_dirs` = type:`list[str]`. A list of input directories for each cycle.  
  - For a **single cycle**, provide a list with one directory.  

`cycles` = type:`list[str]`. A list of cycles corresponding to the input directories.  
  - For a **single cycle**, provide a list with one cycle.

`output_dir_prefix` = type:`str`. This will be the path where you want the preprocessing output to be saved. Ideally, this should be associated with some type of unique project identifier. The format of this variable is `str`. Subfolders for each one of the scanned regions will be created as `R1`, `R2`, etc...

`mode` = type:`str`. 
- `'tif_autosaved'` → autosaved TIFF files in Leica microscope  
- `'tif_exported'` → exported TIFF files in Leica microscope  
- `'lif'` → LIF files in Leica microscope
- `'czi'` → CZI files in Zeiss microscope
- `'nd2'` → Nd2 files in Nikon microscope

`deconvolution_method` = type:`str | None`. 
- `'redlionfish'` → GPU-based  
- `'deconwolf'` → CPU-based (requires Deconwolf executable, see `deconwolf_install.txt`)  
- `None` → skip deconvolution  


`PSF_metadata` = type:`dict | None`.  Refer to the example above. Metadata for the construction of a synthetic PSF. Default is `None`. This can be skipped if `deconvolution_method=None`.

`mip` = type:`bool`, default:`True`. Specifies if the deconvolved images need to be maximum projected (default = True). If `False` the deconvolved stacks are saved, however we do our ISS analysis in 2D so there's almost never a good reason to save the stack.

`align_channel` = type:`int`, default:`4`. The channel used for alignment across cycles (typically DAPI). Refers to the channel index in acquisition order. In Leica setup, DAPI is the 5th channel → set `align_channel=4` (Python is zero-indexed).  

`n_total_cycles` = type:`int`.  Specifies the **total number of expected cycles** for the experiment.  
- This parameter is **mainly important for the stitch and align functions**, which need to know the full number of cycles to correctly align regions across the dataset.  
- The rest of the pipeline (e.g., deconvolve_leica, mipped_to_OME_tiffs) does **not** require all cycles to run correctly.  
- Even if only a subset of cycles is provided in `cycles`, the pipeline uses `n_total_cycles` to maintain consistency in alignment.  
 
>  **Warning:** If `n_total_cycles` is set incorrectly, stitching and alignment may fail or produce shifted/misaligned regions across cycles.  

> **Best practice:** Always set `n_total_cycles` to the **total planned cycles** in the experiment, even if you are only processing a subset of them at a given time.  

`tile_dimension` = type:`int`, default:`6000`.  The number of pixels to tile your images into during the reslicing process. Example: `6000` → resliced images of shape `6000x6000`. 





### Time to start processing!
Now that you have read **ALL the instructions above**, you are ready to process your files. 


In [4]:
input_dirs = ['/path/to/cycle1/',
              '/path/to/cycle2/',
              '/path/to/cycle3/',
              '/path/to/cycle4/',
              '/path/to/cycle5/']

cycles = [1,2,3,4,5]

output_dir_prefix='/path/to/my/output/folder/'

In [5]:
input_dirs = ['/mnt/DATA/ext_home/saga/moldia/saga/zeiss/cycle3',
             '/mnt/DATA/ext_home/saga/moldia/saga/zeiss/cycle6']
cycles = [3,6]
output_dir_prefix = '/mnt/DATA/ext_home/saga/moldia/saga/output_zeiss/'



In [6]:
pp.preprocessing_main(
                input_dirs,
                cycles,
                output_dir_prefix,
                mode='czi',
                deconvolution_method='redlionfish',
                PSF_metadata=PSF_metadata,
                align_channel=4,
                n_total_cycles=1,
                tile_dimension=6000)

Deconvolution and mipping
Processing Cycle 3
Processing directory:  /mnt/DATA/ext_home/saga/moldia/saga/zeiss/cycle3
Mode: czi                                                                       
Deconvolution method: redlionfish                                               
Using CZI file: 250622_FC_OT6_cycle3(1).czi
CZI dims: {'X': 2048, 'Y': 2048, 'Z': 33, 'C': 5, 'M': 14, 'S': 1, 'B': 1}
Regions to be processed: ['Region_1']
Processing R1
14 tiles, 33 Z-slices, 5 channels, image size 2048 × 2048
Expected number of output files in /mnt/DATA/ext_home/saga/moldia/saga/output_zeiss/R1/preprocessing/Cycle3/1_mipped: 70 (14 tiles × 5 channels)
12 tile(s) have missing outputs. Proceeding with processing only these.
Extracting metadata
Calculating the PSF
Generating PSF for channel 0
Generating PSF for channel 1
Generating PSF for channel 2
Generating PSF for channel 3
Generating PSF for channel 4
Generating PSF for channel 5
Number of tiles to process: 14


[Cycle 3] Tile 10, Channel 0...


RuntimeError: Not enough data read at offset 16294813600 -> requested: 880 bytes, actually got 0 bytes.

#### Access to individual functions for Leica processing

Instead of running the main function as outlined above, we can also choose to run the step by step subfunctions one at a time. 

**These functions processes multiple cycles and regions at a time.** However, you need to specify a list of the cycles corresponding to the list of input directories. 

Let's have a quick look at what each function does.


### `deconvolve_and_mip`

`deconvolve_and_mip` is the function to **deconvolve and maximum-project the images from the input folders**. It takes the following arguments, which mirror the same arguments of the main function:

`input_dirs` = type:`list[str]`. A list of input directories for each cycle.  
  - For a **single cycle**, provide a list with one directory.  

`cycles` = type:`list[str]`. A list of cycles corresponding to the input directories.  
  - For a **single cycle**, provide a list with one cycle.

`output_dir_prefix` = type:`str`. This will be the path where you want the preprocessing output to be saved. Ideally, this should be associated with some type of unique project identifier. The format of this variable is `str`. Subfolders for each one of the scanned regions will be created as `R1`, `R2`, etc...

`mode` = type:`str`. 
- `'tif_autosaved'` → autosaved TIFF files in Leica microscope  
- `'tif_exported'` → exported TIFF files in Leica microscope  
- `'lif'` → LIF files in Leica microscope
- `'czi'` → CZI files in Zeiss microscope
- `'nd2'` → Nd2 files in Nikon microscope
  

`deconvolution_method` = type:`str | None`. 
- `'redlionfish'` → GPU-based  
- `'deconwolf'` → CPU-based (requires Deconwolf executable, see `deconwolf_install.txt`)  
- `None` → skip deconvolution  


`PSF_metadata` = type:`dict | None`.  Refer to the example above. Metadata for the construction of a synthetic PSF. Default is `None`. This can be skipped if `deconvolution_method=None`.

`mip` = type:`bool`, default:`True`. Specifies if the deconvolved images need to be maximum projected (default = True). If `False` the deconvolved stacks are saved, however we do our ISS analysis in 2D so there's almost never a good reason to save the stack.


The function outputs deconvolved and mipped images per cycle into the `/preprocessing/Cycle{cycle}/1_mipped/` subfolder, **as well as a list of region directories that can be used in the following functions below (same for all cycles)**.



In [ ]:
input_dirs = ['/path/to/cycle1/',
              '/path/to/cycle2/',
              '/path/to/cycle3/',
              '/path/to/cycle4/',
              '/path/to/cycle5/']

cycles = [1,2,3,4,5]

output_dir_prefix='/path/to/my/output/folder/'

In [ ]:
region_directories = pp.deconvolve_and_mip(
                        input_dirs,
                        cycles,
                        output_dir_prefix, 
                        mode='tif_autosaved',
                        deconvolution_method='redlionfish',
                        PSF_metadata=PSF_metadata
                        )

### `mipped_to_OME_tiffs`
`mipped_to_OME_tiffs` is the function that **takes the projected images across channels and wraps them into a single OMEtiff per imaging cycle**. This steps organises the files corresponding to each imaging cycles in a specific way within a single file and requires the parsing of a Metadata file to arrange correctly the images in xy space. 

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories` = type:`list[str]`  
List of directories for each region to be processed.  
The output generated by `deconvolve_leica` can be used directly as input here.  
 
`cycles` = type:`list[str]`. A list of cycles corresponding to the input directories.  
  - For a **single cycle**, provide a list with one cycle.

Mipped images need to be accessible in `/preprocessing/Cycle{cycle}/1_mipped/` subfolder in each region directory.

The function outputs one OMEtiff file per cycle into the `/preprocessing/Cycle{cycle}/2_ome_tiffs/` subfolder.



In [ ]:
pp.mipped_to_OME_tiffs(
    region_directories,
    cycles
    )

### `align_and_stitch`

This function runs `ashlar`, a package for image stitching and cycle alignment. The function uses the OME_tiffs files as an input, takes as input a channel number (normally the DAPI, see above) and on that channels performs all the alignment and stitching operations.

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories` = type:`list[str]`  
List of directories for each region to be processed.  
The output generated by `deconvolve_leica` can be used directly as input here.  
 
`cycles` = type:`list[str]`. A list of cycles corresponding to the input directories.  
  - For a **single cycle**, provide a list with one cycle.

`align_channel` = type:`int`, default:`4`. The channel used for alignment across cycles (typically DAPI). Refers to the channel index in acquisition order. In Leica setup, DAPI is the 5th channel → set `align_channel=4` (Python is zero-indexed).  

`n_total_cycles` = type:`int`.  Specifies the **total number of expected cycles** for the experiment.  
- This parameter is **mainly important for the stitch and align functions**, which need to know the full number of cycles to correctly align regions across the dataset.  
- The rest of the pipeline (e.g., deconvolve_leica, mipped_to_OME_tiffs) does **not** require all cycles to run correctly.  
- Even if only a subset of cycles is provided in `cycles`, the pipeline uses `n_total_cycles` to maintain consistency in alignment.  
 
>  **Warning:** If `n_total_cycles` is set incorrectly, stitching and alignment may fail or produce shifted/misaligned regions across cycles.  

> **Best practice:** Always set `n_total_cycles` to the **total planned cycles** in the experiment, even if you are only processing a subset of them at a given time.  


OMEtiff file needs to be accessible in `/preprocessing/Cycle{cycle}/2_ome_tiff/` subfolder in each region directory.

The function outputs one stitched file per cycle and channel into the `/preprocessing/Cycle{cycle}/3_stitched/` subfolder

In [ ]:
pp.align_and_stitch(
    region_directories,
    cycles,
    align_channel=4,
    n_total_cycles=5
)

### `retile_stitched_images`
In this function the stitched images are re-tiled according to a user-specified size.
The reason for this is that stitched images are too big to be decoded directly and we prefer to decode them in tiles. This has several advantages, most notably that the pipeline would work also on laptops or non-powerful computers. The idea tile size is 4000-6000, but larger or smaller are also fine depending on the computer.

It takes the following arguments, which mirror the same arguments of the main function:

`region_directories` = type:`list[str]`  
List of directories for each region to be processed.  
The output generated by `deconvolve_leica` can be used directly as input here.  
 
`cycles` = type:`list[str]`. A list of cycles corresponding to the input directories.  
  - For a **single cycle**, provide a list with one cycle.
    
`tile_dimension` = type:`int`, default:`6000`.  The number of pixels to tile your images into during the reslicing process. Example: `6000` → resliced images of shape `6000x6000`. 

The stitched files need to be accessible in `/preprocessing/Cycle{cycle}/3_ome_tiffs/` subfolder in each region directory.

The function outputs one stitched file per cycle and channel into the `/preprocessing/Cycle{cycle}/4_retiled/` subfolder

In [ ]:
pp.retile_stitched_images(
    region_directories,
    cycles,
    tile_dimension=6000
)